In [2]:
!pip install -q datasets transformers pandas tqdm

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import  BertTokenizer, BertModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from datasets import load_dataset
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np

In [4]:
CONFIG = {
    "snli_samples": 100_000,
    "mnli_samples": 100_000,
    "model_name": "bert-base-uncased",
    "max_length": 64,       # Shorter sequence length saves memory
    "batch_size": 64,       # Max for T4
    "accumulation_steps": 2, # Effective batch size = 128
    "epochs": 2,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.1,    # 10% of steps for warmup
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

print(f"Using device: {CONFIG['device']}")

Using device: cuda


In [5]:
def create_triplets_from_dataset(dataset_name, n_samples, config_name=None, split="train"):
    """
    Loads a dataset, converts to pandas, and groups to find (A, P, N) triplets.
    Handles both single-config (like 'snli') and multi-config (like 'glue') datasets.
    """
    print(f"Loading {dataset_name}" + (f" (config: {config_name})" if config_name else "") + "...")

    # Load a slice of the dataset
    # The 'config_name' argument is correctly passed to load_dataset
    dataset = load_dataset(dataset_name, config_name, split=f'{split}[:{n_samples}]')
    df = dataset.to_pandas()

    # Filter out neutral (label 1) and any NaNs
    df = df[df['label'] != 1]
    df = df.dropna(subset=['premise', 'hypothesis'])

    print("Grouping by premise to find (Anchor, Positive, Negative) pairs...")
    grouped = df.groupby('premise')
    triplets = []

    desc = f"Processing {dataset_name}" + (f" ({config_name})" if config_name else "")
    for premise, group in tqdm(grouped, desc=desc):
        positives = group[group['label'] == 0]['hypothesis'].tolist()
        negatives = group[group['label'] == 2]['hypothesis'].tolist()

        # Only add if we have at least one of each
        if positives and negatives:
            # We only need one of each for this triplet
            triplets.append((premise, positives[0], negatives[0]))

    print(f"Found {len(triplets)} triplets in {dataset_name}" + (f" ({config_name})" if config_name else "") + ".")
    return triplets

In [6]:
snli_triplets = create_triplets_from_dataset(
    'snli',
    n_samples=CONFIG['snli_samples']
)

Loading snli...


README.md: 0.00B [00:00, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

Grouping by premise to find (Anchor, Positive, Negative) pairs...


Processing snli:   0%|          | 0/27436 [00:00<?, ?it/s]

Found 27195 triplets in snli.


In [7]:
mnli_triplets = create_triplets_from_dataset(
    'glue',
    n_samples=CONFIG['mnli_samples'],
    config_name='mnli'
)

Loading glue (config: mnli)...


README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

Grouping by premise to find (Anchor, Positive, Negative) pairs...


Processing glue (mnli):   0%|          | 0/59295 [00:00<?, ?it/s]

Found 9242 triplets in glue (mnli).


In [8]:
all_triplets = snli_triplets + mnli_triplets
print(f"\nTotal aligned triplets for training: {len(all_triplets)}")


Total aligned triplets for training: 36437


In [9]:
class TripletDataset(Dataset):
  def __init__(self, triplets, tokenizer, max_length=CONFIG['max_length']):
    self.triplets = triplets
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.triplets)

  def __getitem__(self, idx):
    anchor, positive, negative = self.triplets[idx]

    return{
        "anchor": anchor,
        "positive": positive,
        "negative": negative,
    }

In [10]:
def collate_fn(batch):
  anchors = [item['anchor'] for item in batch]
  positives = [item['positive'] for item in batch]
  negatives = [item['negative'] for item in batch]

  tok_anchors = tokenizer(anchors, padding=True, truncation=True, max_length=CONFIG['max_length'], return_tensors='pt')
  tok_positives = tokenizer(positives, padding=True, truncation=True, max_length=CONFIG['max_length'], return_tensors='pt')
  tok_negatives = tokenizer(negatives, padding=True, truncation=True, max_length=CONFIG['max_length'], return_tensors='pt')

  return{
      "anchor": tok_anchors,
      "positive": tok_positives,
      "negative": tok_negatives,
  }


tokenizer = BertTokenizer.from_pretrained(CONFIG['model_name'])
train_dataset = TripletDataset(all_triplets, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [11]:
class SimCSEModel(nn.Module):
    def __init__(self, model_name=CONFIG['model_name']):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask, token_type_ids):
        # Get BERT outputs
        outputs = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)

        # Use the standard BERT pooler output ([CLS] token + Linear/Tanh)
        pooled_output = outputs.pooler_output

        # Normalize the embedding to have unit length
        # This is crucial for contrastive learning with cosine similarity
        return F.normalize(pooled_output, p=2, dim=1)

model = SimCSEModel().to(CONFIG['device'])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [12]:
model

SimCSEModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [13]:
def constrastive_loss(anchor, positive, negative, temprature=0.05):

  sim_ap = F.cosine_similarity(anchor, positive, dim = 1)
  sim_an = F.cosine_similarity(anchor, negative, dim = 1)

  logits = torch.stack([sim_ap, sim_an], dim=1)

  logits = logits / temprature

  labels = torch.zeros(logits.shape[0], dtype=torch.long, device=logits.device)

  return nn.CrossEntropyLoss()(logits, labels)

loss_fn = constrastive_loss

In [14]:
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
total_steps = (len(train_loader) // CONFIG['accumulation_steps']) * CONFIG['epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [15]:
for epoch in range(CONFIG['epochs']):
    print(f"\n--- Epoch {epoch + 1}/{CONFIG['epochs']} ---")

    total_loss = 0
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc="Training")

    for i, batch in progress_bar:
        # 1. Move all parts of the batch to the device
        anc_inputs = {k: v.to(CONFIG['device']) for k, v in batch['anchor'].items()}
        pos_inputs = {k: v.to(CONFIG['device']) for k, v in batch['positive'].items()}
        neg_inputs = {k: v.to(CONFIG['device']) for k, v in batch['negative'].items()}

        # 2. Get embeddings for all three
        emb_a = model(**anc_inputs)
        emb_p = model(**pos_inputs)
        emb_n = model(**neg_inputs)

        # 3. Compute loss
        loss = loss_fn(emb_a, emb_p, emb_n)

        # 4. Gradient Accumulation: Normalize loss and backpropagate
        loss = loss / CONFIG['accumulation_steps']
        loss.backward()

        # 5. Gradient Accumulation: Step and clear
        if (i + 1) % CONFIG['accumulation_steps'] == 0:
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CONFIG['accumulation_steps'] # De-normalize loss for logging

        # Update progress bar
        progress_bar.set_postfix({'loss': total_loss / (i + 1)})

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")

print("=== Training Finished ===")


--- Epoch 1/2 ---


Training:   0%|          | 0/570 [00:00<?, ?it/s]

Epoch 1 Average Loss: 0.4272

--- Epoch 2/2 ---


Training:   0%|          | 0/570 [00:00<?, ?it/s]

Epoch 2 Average Loss: 0.1637
=== Training Finished ===


In [16]:
import pandas as pd
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
import numpy as np

# 1. Load the FULL STSb test set (the standard benchmark)
sts = load_dataset("stsb_multi_mt", name="en", split="test")
sts = pd.DataFrame(sts)
print(f"Loaded {len(sts)} samples from STSb test set.")

# 2. Ensure model is in evaluation mode
model.eval()

# 3. Define the embedding function
def embed_texts(texts):
    if isinstance(texts, pd.Series):
        texts = texts.tolist()

    # Tokenize
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors='pt',
        max_length=CONFIG['max_length']
    ).to(CONFIG['device'])

    # Run model
    with torch.no_grad():
        # ** Use the corrected forward pass **
        return model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            token_type_ids=enc['token_type_ids']
        ).cpu()

print("Generating embeddings for sentence 1...")
emb1 = embed_texts(sts['sentence1'])

print("Generating embeddings for sentence 2...")
emb2 = embed_texts(sts['sentence2'])

# 4. Compute cosine similarity between the two embedding sets
similarities = cosine_similarity(emb1, emb2).diagonal()

# 5. Evaluate Spearman correlation against human labels
# The 'similarity_score' from the dataset is normalized 0-5.
# We just need to scale it to 0-1 for a 1:1 comparison.
corr = spearmanr(similarities, sts['similarity_score'])

print("\n--- Evaluation Complete ---")
print(f"Spearman Correlation: {corr.correlation:.4f}")

Loaded 1379 samples from STSb test set.
Generating embeddings for sentence 1...
Generating embeddings for sentence 2...

--- Evaluation Complete ---
Spearman Correlation: 0.6280


In [17]:
import os

# Define a directory name to save your model
output_dir = "./my_v2_simcse_model"

# Create the directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"Saving model to {output_dir}...")

# 1. Save the BERT model (the part with the learned weights)
# We save `model.bert` because that is the underlying Hugging Face model
model.bert.save_pretrained(output_dir)

# 2. Save the Tokenizer
# This saves the vocab.txt and tokenizer config
tokenizer.save_pretrained(output_dir)

print("Model and tokenizer saved successfully!")

Saving model to ./my_v2_simcse_model...
Model and tokenizer saved successfully!
